# Clase 17 — Patrones de Diseño

En esta clase aprenderemos los **patrones de diseño** clásicos (GoF) implementados en Python. Los patrones son soluciones reutilizables a problemas recurrentes en el diseño de software.

**Contenido de esta clase:**
1. ¿Qué son los patrones de diseño?
2. Singleton
3. Factory Method
4. Observer
5. Strategy
6. Decorator (patrón)
7. Cuándo usar cada patrón
8. Anti-patrones
9. Ejercicio práctico


## 1. ¿Qué son los patrones de diseño?

Los **patrones de diseño** son soluciones probadas a problemas comunes en el diseño de software orientado a objetos. No son código específico, sino **plantillas** que se pueden adaptar a diferentes situaciones.

### Las 4 categorías (GoF)

| Categoría | Propósito | Ejemplos |
|---|---|---|
| **Creacionales** | Cómo crear objetos | Singleton, Factory, Builder |
| **Estructurales** | Cómo componer clases | Adapter, Decorator, Facade |
| **De comportamiento** | Cómo interactúan los objetos | Observer, Strategy, Command |
| **Concurrencia** | Cómo manejar hilos | (no los veremos) |


## 2. Singleton

**Propósito:** Garantizar que una clase tenga **una sola instancia** y proporcionar un punto de acceso global.

In [ ]:
class Singleton:
    """Implementación básica de Singleton."""
    _instancia = None
    
    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
        return cls._instancia

# Verificar que solo hay una instancia
s1 = Singleton()
s2 = Singleton()
print(f"¿Misma instancia? {s1 is s2}")
print(f"id(s1) == id(s2): {id(s1) == id(s2)}")


In [ ]:
# Singleton más completo con inicialización
class Configuracion:
    """Singleton para configuración global de la aplicación."""
    _instancia = None
    _inicializado = False
    
    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
        return cls._instancia
    
    def __init__(self):
        if not Configuracion._inicializado:
            self.base_datos = "mi_app.db"
            self.debug = False
            self.version = "1.0.0"
            Configuracion._inicializado = True
    
    def __repr__(self):
        return f"Config(version={self.version}, debug={self.debug})"

# Uso
config1 = Configuracion()
config1.debug = True  # Cambiar en una instancia
config2 = Configuracion()  # Misma instancia
print(f"config1: {config1}")
print(f"config2: {config2}")
print(f"¿Comparten estado? {config1.debug == config2.debug}")


## 3. Factory Method

**Propósito:** Crear objetos sin especificar la clase exacta, delegando la decisión a una subclase o método.


In [ ]:
from abc import ABC, abstractmethod
# Sin Factory: acoplamiento directo
# El cliente tiene que conocer las clases concretas
""" 
class NotificacionEmail:
    def enviar(self, mensaje):
        return f"Email enviado: {mensaje}"

class NotificacionSMS:
    def enviar(self, mensaje):
        return f"SMS enviado: {mensaje}"
 """
# Con Factory: el cliente NO conoce las clases concretas
# Solo pide "dame un notificador de tipo X" y usa la interfaz común
class Notificador(ABC):
    """Interfaz común: todos los notificadores saben `enviar(mensaje)`."""
    @abstractmethod
    def enviar(self, mensaje) -> str: ...

class NotificacionEmail(Notificador):
    def enviar(self, mensaje):
        return f"Email enviado: {mensaje}"

class NotificacionSMS(Notificador):
    def enviar(self, mensaje):
        return f"SMS enviado: {mensaje}"

class NotificacionFactory:
    """Factory Method: crea objetos sin especificar la clase exacta.
    
    El cliente obtiene un OBJETO, no el resultado de un método.
    Decide cuándo y cómo usar el objeto después.
    """
    
    @staticmethod
    def crear(tipo):
        """Crea y RETORNA un objeto notificador según el tipo."""
        if tipo == "email":
            return NotificacionEmail()
        elif tipo == "sms":
            return NotificacionSMS()
        else:
            raise ValueError(f"Tipo desconocido: {tipo}")

# Uso correcto del Factory:
# 1. Pedir el objeto
notificador = NotificacionFactory.crear("email")
# 2. Usar el objeto DESPUÉS (el cliente decide cuándo)
resultado = notificador.enviar("Hola desde email")
print(resultado)

# Comparación: con Factory desacoplas la creación del uso
# Sin Factory, tendrías que hacer:
# resultado = NotificacionEmail().enviar("Hola")  # Acoplado
# Con Factory:
# notificador = NotificacionFactory.crear("email")  # Desacoplado
# notificador.enviar("Hola")  # Uso posterior

In [ ]:
# Factory más robusto: registro dinámico
# Permite agregar nuevos tipos sin modificar el Factory
class Notificador(ABC):
    @abstractmethod
    def enviar(self, mensaje) -> str: ...

class NotificacionEmail(Notificador):
    def enviar(self, mensaje):
        return f"Email: {mensaje}"

class NotificacionSMS(Notificador):
    def enviar(self, mensaje):
        return f"SMS: {mensaje}"

class NotificacionPush(Notificador):
    def __init__(self, device_id):
        self.device_id = device_id
    def enviar(self, mensaje):
        return f"Push a {self.device_id}: {mensaje}"

class NotificacionRegistry:
    """Factory con registro dinámico de tipos.
    
    El Factory guarda CLASES (o factories) y las usa para crear
    instancias cuando se le pide. Devuelve OBJETOS, no strings.
    """
    
    _creadores = {}
    
    @classmethod
    def registrar(cls, tipo, creador):
        """Registra una clase o función factory para un tipo."""
        cls._creadores[tipo] = creador
    
    @classmethod
    def crear(cls, tipo, *args, **kwargs):
        """Crea y RETORNA un objeto del tipo solicitado."""
        creador = cls._creadores.get(tipo)
        if not creador:
            raise ValueError(f"Tipo no registrado: {tipo}")
        return creador(*args, **kwargs)

# Registrar las CLASES (no lambdas que ejecutan la lógica)
NotificacionRegistry.registrar("email", NotificacionEmail)
NotificacionRegistry.registrar("sms", NotificacionSMS)
# Push requiere un argumento, lo registramos con un lambda que capture device_id
notificador_push_default = NotificacionPush("device-abc123")
NotificacionRegistry.registrar("push", lambda: notificador_push_default)

# Uso correcto: el Factory devuelve OBJETOS
notif_email = NotificacionRegistry.crear("email")
notif_sms = NotificacionRegistry.crear("sms")
notif_push = NotificacionRegistry.crear("push")

# El cliente decide cuándo usar los objetos
print(notif_email.enviar("Hola desde email"))
print(notif_sms.enviar("Hola desde SMS"))
print(notif_push.enviar("Hola desde push"))

# Intentar tipo no registrado
try:
    NotificacionRegistry.crear("telegram")
except ValueError as e:
    print(f"\nError esperado: {e}")

## 4. Observer

**Propósito:** Notificar a múltiples objetos cuando el estado de otro objeto cambia. Es la base de los sistemas de eventos.


In [ ]:
# Implementación del patrón Observer
class Sujeto:
    """Clase base para objetos observables."""
    
    def __init__(self):
        self._observadores = []
    
    def agregar_observador(self, observador):
        self._observadores.append(observador)
    
    def eliminar_observador(self, observador):
        self._observadores.remove(observador)
    
    def notificar(self, evento):
        for observador in self._observadores:
            observador.actualizar(evento)

class SensorTemperatura(Sujeto):
    """Sensor que notifica cuando cambia la temperatura."""
    
    def __init__(self, nombre):
        super().__init__()
        self.nombre = nombre
        self._temperatura = 0
    
    @property
    def temperatura(self):
        return self._temperatura
    
    @temperatura.setter
    def temperatura(self, valor):
        self._temperatura = valor
        self.notificar(f"{self.nombre}: {valor}°C")

# Observadores
class Display:
    def actualizar(self, evento):
        print(f"  [Display] {evento}")

class Logger:
    def actualizar(self, evento):
        print(f"  [Logger] Registrado: {evento}")

class Alerta:
    def __init__(self, umbral):
        self.umbral = umbral
    
    def actualizar(self, evento):
        if "°C" in evento:
            temp = float(evento.split(":")[-1].replace("°C", "").strip())
            if temp > self.umbral:
                print(f"  [ALERTA] ¡Temperatura crítica: {temp}°C!")


In [ ]:
# Usar el sistema de observación
sensor = SensorTemperatura("Sala")
display = Display()
logger = Logger()
alerta = Alerta(umbral=25)

# Registrar observadores
sensor.agregar_observador(display)
sensor.agregar_observador(logger)
sensor.agregar_observador(alerta)

# Cambiar temperatura (dispara notificaciones)
print("Temperatura 20°C:")
sensor.temperatura = 20

print("\nTemperatura 30°C:")
sensor.temperatura = 30


## 5. Strategy

**Propósito:** Definir una familia de algoritmos, encapsular cada uno, y hacerlos intercambiables.


In [ ]:
# Estrategias de ordenación (Strategy)
from abc import ABC, abstractmethod

class EstrategiaOrdenamiento(ABC):
    """Interfaz para estrategias de ordenación."""
    
    @abstractmethod
    def ordenar(self, lista):
        pass

class OrdenamientoBurbuja(EstrategiaOrdenamiento):
    def ordenar(self, lista):
        datos = lista.copy()
        n = len(datos)
        for i in range(n):
            for j in range(n - 1 - i):
                if datos[j] > datos[j + 1]:
                    datos[j], datos[j + 1] = datos[j + 1], datos[j]
        return datos

class OrdenamientoRapido(EstrategiaOrdenamiento):
    def ordenar(self, lista):
        if len(lista) <= 1:
            return lista
        pivote = lista[len(lista) // 2]
        return ([x for x in lista if x < pivote] + 
                [x for x in lista if x == pivote] + 
                [x for x in lista if x > pivote])

class OrdenamientoPython(EstrategiaOrdenamiento):
    """Usa sorted() de Python (Timsort)."""
    def ordenar(self, lista):
        return sorted(lista)


In [ ]:
# Contexto que usa las estrategias
class GestorOrdenamiento:
    """Contexto que usa una estrategia de ordenación."""
    
    def __init__(self, estrategia: EstrategiaOrdenamiento):
        self.estrategia = estrategia
    
    def ordenar(self, lista):
        return self.estrategia.ordenar(lista)

# Probar diferentes estrategias
datos = [64, 34, 25, 12, 22, 11, 90]

gestor = GestorOrdenamiento(OrdenamientoBurbuja())
print(f"Burbuja: {gestor.ordenar(datos)}")

gestor = GestorOrdenamiento(OrdenamientoRapido())
print(f"Quicksort: {gestor.ordenar(datos)}")

gestor = GestorOrdenamiento(OrdenamientoPython())
print(f"Python sorted: {gestor.ordenar(datos)}")


## 6. Decorator (patrón)

**Propósito:** Añadir funcionalidad a objetos dinámicamente sin modificar su clase. Es un patrón **estructural** (diferente del syntax sugar `@decorador` de Python).


In [ ]:
# Patrón Decorator: añade funcionalidad envolviendo objetos

class Notificador:
    """Interfaz base para notificadores."""
    def enviar(self, mensaje):
        pass

class NotificadorBasico(Notificador):
    def enviar(self, mensaje):
        return f"Básico: {mensaje}"

class NotificadorDecorator(Notificador):
    """Base para decoradores de notificador."""
    def __init__(self, notificador):
        self._notificador = notificador
    
    def enviar(self, mensaje):
        return self._notificador.enviar(mensaje)

class ConLogging(NotificadorDecorator):
    """Decorador que añade logging."""
    def enviar(self, mensaje):
        resultado = self._notificador.enviar(mensaje)
        print(f"  [LOG] Enviado: {mensaje}")
        return resultado

class ConEncriptacion(NotificadorDecorator):
    """Decorador que añade encriptación (simulada)."""
    def enviar(self, mensaje):
        mensaje_encriptado = f"[ENC]{mensaje}[/ENC]"
        return self._notificador.enviar(mensaje_encriptado)

class ConCompresion(NotificadorDecorator):
    """Decorador que añade compresión (simulada)."""
    def enviar(self, mensaje):
        mensaje_comprimido = f"[ZIP({len(mensaje)})]{mensaje}"
        return self._notificador.enviar(mensaje_comprimido)


In [ ]:
# Componer decoradores
notificador = NotificadorBasico()
print(f"--- Básico ---\n{notificador.enviar('Hola')}\n")

# Añadir logging
notificador = ConLogging(NotificadorBasico())
print(f"--- Con logging ---\n{notificador.enviar('Hola')}\n")

# Combinar: logging + encriptación + compresión
notificador = ConLogging(ConEncriptacion(ConCompresion(NotificadorBasico())))
print(f"--- Todo combinado ---")
print(notificador.enviar("Mensaje secreto"))


## 7. Cuándo usar cada patrón

| Patrón | Úsalo cuando... | Ejemplo real |
|---|---|---|
| **Singleton** | Necesitas exactamente una instancia (config, logger) | Configuración de app |
| **Factory** | Quieres delegar la creación de objetos | Crear conexiones a BD |
| **Observer** | Múltiples objetos deben reaccionar a cambios | Sistema de eventos, UI |
| **Strategy** | Tienes varios algoritmos intercambiables | Ordenamiento, compresión |
| **Decorator** | Quieres añadir funcionalidad dinámicamente | Middleware, logging |


## 8. Anti-patrones comunes

**Anti-patrones** son soluciones "obvias" que parecen buenas pero causan problemas a largo plazo.


In [ ]:
# ❌ Anti-patrón: "God Object"
# Un objeto que hace TODO
class GestorAplicacion:
    """❌ Hace demasiado: UI, lógica, datos, red, etc."""
    def conectar_bd(self): pass
    def renderizar_ui(self): pass
    def enviar_email(self): pass
    def calcular_impuestos(self): pass
    def autenticar_usuario(self): pass
    # ... 500 métodos más

# ✓ Mejor: separar responsabilidades
class ConexionBD: pass
class RenderizadorUI: pass
class ServicioEmail: pass
class CalculadoraImpuestos: pass
class Autenticador: pass
# Cada clase tiene una responsabilidad clara


In [ ]:
# ❌ Anti-patrón: "Spaghetti Code" (funciones que se llaman entre sí sin estructura)
def funcion_a():
    funcion_b()  # Llama a b
def funcion_b():
    funcion_c()  # Llama a c
def funcion_c():
    funcion_a()  # Llama a a (ciclo!)

# ✓ Mejor: estructura clara con clases y eventos
class Sistema:
    def __init__(self):
        self.eventos = []
    def emitir(self, evento):
        self.eventos.append(evento)
    def procesar(self):
        for evento in self.eventos:
            self.manejar(evento)
    def manejar(self, evento):
        # Lógica clara aquí
        pass


## 9. Ejercicio Práctico: Sistema de Notificaciones con Patrones

**Contexto:** Vas a crear un sistema de notificaciones que use **Singleton** (configuración), **Factory** (crear notificadores) y **Observer** (notificar a múltiples destinos).

**Instrucciones:**

Crea un archivo `sistema_notificaciones.py` con:

1. `Configuracion` (Singleton) — Configuración global con email, SMS, push habilitados.
2. `NotificadorFactory` — Crea el notificador apropiado según el tipo.
3. `SistemaNotificaciones` (Observer) — Permite suscribir "canales" (log, display, etc.) y notifica a todos cuando llega un mensaje.
4. `NotificadorEmail`, `NotificadorSMS` — Implementaciones concretas.
5. `CanalLog`, `CanalDisplay` — Observadores que reciben notificaciones.

**Pistas:**
- Singleton: usa `__new__` para garantizar una sola instancia.
- Factory: usa un diccionario de clases o funciones.
- Observer: usa una lista de observadores y notifícalos cuando llegue un mensaje.

**Restricción del ejercicio:**
- No uses librerías externas.
- Implementa los patrones desde cero.


In [ ]:
"""
Solución al Ejercicio Práctico
Este código debe ir en un archivo sistema_notificaciones.py
"""

class Configuracion:
    """Singleton: configuración global del sistema."""
    _instancia = None
    
    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.email_habilitado = True
            cls._instancia.sms_habilitado = True
            cls._instancia.push_habilitado = True
        return cls._instancia

class Notificador:
    """Interfaz base."""
    def enviar(self, mensaje, destinatario):
        raise NotImplementedError

class NotificadorEmail(Notificador):
    def enviar(self, mensaje, destinatario):
        return f"📧 Email a {destinatario}: {mensaje}"

class NotificadorSMS(Notificador):
    def enviar(self, mensaje, destinatario):
        return f"📱 SMS a {destinatario}: {mensaje}"

class NotificadorFactory:
    """Factory: crea el notificador apropiado."""
    _notificadores = {
        'email': NotificadorEmail,
        'sms': NotificadorSMS,
    }
    
    @classmethod
    def crear(cls, tipo):
        notificador_cls = cls._notificadores.get(tipo)
        if not notificador_cls:
            raise ValueError(f"Tipo desconocido: {tipo}")
        return notificador_cls()

class SistemaNotificaciones:
    """Observer: notifica a múltiples canales."""
    
    def __init__(self):
        self.canales = []
    
    def suscribir(self, canal):
        self.canales.append(canal)
    
    def notificar_todos(self, evento):
        for canal in self.canales:
            canal.recibir(evento)

class CanalLog:
    def recibir(self, evento):
        print(f"  [LOG] {evento}")

class CanalDisplay:
    def __init__(self):
        self.ultimo_evento = None
    def recibir(self, evento):
        self.ultimo_evento = evento
        print(f"  [DISPLAY] Mostrando: {evento}")


In [ ]:
# Probar el sistema completo
config = Configuracion()
print(f"Email habilitado: {config.email_habilitado}")

sistema = SistemaNotificaciones()
sistema.suscribir(CanalLog())
display = CanalDisplay()
sistema.suscribir(display)

# Crear notificador y enviar
email = NotificadorFactory.crear('email')
sms = NotificadorFactory.crear('sms')

# Enviar notificaciones
resultado = email.enviar("Bienvenido al curso", "ana@uni.edu")
sistema.notificar_todos(resultado)

resultado = sms.enviar("Tu código es 1234", "+525512345678")
sistema.notificar_todos(resultado)

# Intentar tipo no válido
try:
    NotificadorFactory.crear('telegram')
except ValueError as e:
    print(f"\nError: {e}")


## Resumen de la clase

En esta clase aprendimos:

1. **Patrones de diseño:** soluciones probadas a problemas recurrentes.
2. **Singleton:** garantiza una sola instancia (config, logger).
3. **Factory Method:** crea objetos sin especificar la clase exacta.
4. **Observer:** notifica a múltiples objetos sobre cambios de estado.
5. **Strategy:** algoritmos intercambiables en tiempo de ejecución.
6. **Decorator (patrón):** añade funcionalidad envolviendo objetos.
7. **Cuándo usar cada patrón:** según el problema y el contexto.
8. **Anti-patrones:** God Object, Spaghetti Code, reinventar la rueda.

**En la siguiente clase** (Clase 18) veremos el **Cierre del Curso**: comprehensions, regex, async/await, Clean Code y herramientas de IA.
